#### Notebook to check the (average)energy dependence of effective area in DL3. Can be used to select an appropriate safe mask in gammapy DL3 analysis

In [ ]:
import glob
from astropy.io import fits

In [ ]:
input_dir = ""

In [ ]:
dl3_file = glob.glob(f"{input_dir}/dl3*.gz")
enlow0 = []  # energy lower bin edges in first dl3 file
enhigh0 = []  # energy higher bin edges in first dl3 file
aeff = [[] for _ in range(len(dl3_file))]  # array of eff area values per file

for i, d3f in enumerate(dl3_file):
    hdul = fits.open(d3f)
    energy_low = hdul[4].data["ENERG_LO"]
    energy_high = hdul[4].data["ENERG_HI"]
    if i == 0:  # fill in energy edges for first files
        enlow0 = energy_low
        enhigh0 = energy_high
    else:  # for all files except first, check if energy edges are the same as for the first file
        if (enlow0 == energy_low).all() and (enhigh0 == energy_high).all():
            pass
        else:
            print("Not same energy binning in all files")
            break
    aeff[i] = hdul[4].data["EFFAREA"]  # fill in array of effective areas


energymid = (enlow0[0] + enhigh0[0]) / 2  # energy bins center
aeffavg = np.mean(
    [aeff[i][0][0] for i in range(len(dl3_file))], axis=0
)  # average (per E bin) of Eff area over all files
aeffmax = np.nanmax(aeffavg)  # Max of average eff area value


plt.plot(energymid, aeffavg)
plt.yscale("log")
plt.xscale("log")
plt.xlabel("E [TeV]")
plt.ylabel("Aeff (average over all the runs) [m2]")
plt.axhline(y=0.01 * aeffmax, label="1% Aeff max", c="r")
plt.axhline(y=0.03 * aeffmax, label="3% Aeff max", c="g")
plt.axhline(y=0.05 * aeffmax, label="5% Aeff max", c="k")
plt.axhline(y=0.1 * aeffmax, label="10% Aeff max", c="orange")
plt.grid(which="both")
plt.legend()